<a href="https://colab.research.google.com/github/FaithChege254/FaithChege254/blob/main/Web_App_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# Sunbird AI - Multilingual Speech Assistant
# ============================================================

# Upgrade pip
!pip -q install --upgrade pip

# Hugging Face ecosystem
!pip -q install transformers accelerate datasets huggingface_hub

# ASR
!pip -q install openai-whisper
!pip -q install librosa soundfile torchaudio

# Translation
!pip -q install deep-translator

# LLM
!pip -q install bitsandbytes

# Web App
!pip -q install gradio

# Utilities
!pip -q install pandas numpy tqdm

# Optional (for recording/audio handling)
!apt-get -qq install ffmpeg

print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES & CONFIGURE ENVIRONMENT
# ============================================================

import os
import gc
import torch
import librosa
import tempfile
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import gradio as gr

from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    pipeline
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# DEVICE CONFIGURATION
# ------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print(" SUNBIRD AI - MULTILINGUAL SPEECH ASSISTANT")
print("=" * 60)

print(f" Device          : {DEVICE}")

if DEVICE == "cuda":
    print(f" GPU             : {torch.cuda.get_device_name(0)}")
    print(f" CUDA Version    : {torch.version.cuda}")
else:
    print("Running on CPU (GPU recommended)")

print(f" PyTorch Version : {torch.__version__}")

print("=" * 60)

# ------------------------------------------------------------
# OPTIONAL PERFORMANCE SETTINGS
# ------------------------------------------------------------

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Environment initialized successfully!")

 SUNBIRD AI - MULTILINGUAL SPEECH ASSISTANT
 Device          : cuda
 GPU             : Tesla T4
 CUDA Version    : 12.8
 PyTorch Version : 2.11.0+cu128
✅ Environment initialized successfully!


In [ ]:
from huggingface_hub import login

login()

In [ ]:
# ============================================================
# CELL 4: PROJECT CONFIGURATION
# ============================================================

import os
import torch

# ------------------------------------------------------------
# APPLICATION INFORMATION
# ------------------------------------------------------------
APP_NAME = "Multilingual Clinical Speech Assistant"
APP_VERSION = "1.0.0"

# ------------------------------------------------------------
# DEVICE CONFIGURATION
# ------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# ------------------------------------------------------------
# MODEL CONFIGURATION
# ------------------------------------------------------------

# Automatic Speech Recognition (ASR)
ASR_MODEL_ID = "Sunbird/asr-whisper-large-v3-salt"

# Translation Service
TRANSLATION_SERVICE = "Google Translate"

# LLM
# Replace this with the exact model from your notebook later
LLM_MODEL_ID = "YOUR_LLM_MODEL"

# Text-to-Speech (TTS)
TTS_MODEL_ID = "facebook/mms-tts-swh"

# ------------------------------------------------------------
# SUPPORTED LANGUAGES
# ------------------------------------------------------------
SUPPORTED_LANGUAGES = {
    "English": "en",
    "Kiswahili": "sw",
    "Kikuyu": "en"
}

DEFAULT_INPUT_LANGUAGE = "Auto Detect"
DEFAULT_OUTPUT_LANGUAGE = "English"

# ------------------------------------------------------------
# AUDIO SETTINGS
# ------------------------------------------------------------
MAX_AUDIO_DURATION = 120  # seconds

SUPPORTED_AUDIO_FORMATS = [
    ".wav",
    ".mp3",
    ".flac",
    ".m4a",
    ".ogg"
]

# ------------------------------------------------------------
# TEMPORARY DIRECTORY
# ------------------------------------------------------------
TEMP_DIR = "temp_audio"
os.makedirs(TEMP_DIR, exist_ok=True)

# ------------------------------------------------------------
# DISPLAY CONFIGURATION
# ------------------------------------------------------------
print("=" * 60)
print(f"🚀 {APP_NAME} v{APP_VERSION}")
print("=" * 60)
print(f"🖥️ Device                : {DEVICE}")
print(f"🎤 ASR Model             : {ASR_MODEL_ID}")
print(f" Translation Service   : {TRANSLATION_SERVICE}")
print(f" LLM                  : {LLM_MODEL_ID}")
print(f" Temp Folder          : {TEMP_DIR}")
print("=" * 60)
print("✅ Configuration Loaded Successfully!")

🚀 Multilingual Clinical Speech Assistant v1.0.0
🖥️ Device                : cuda
🎤 ASR Model             : Sunbird/asr-whisper-large-v3-salt
 Translation Service   : Google Translate
 LLM                  : YOUR_LLM_MODEL
 Temp Folder          : temp_audio
✅ Configuration Loaded Successfully!


In [ ]:
# ============================================================
# CELL 5: LOAD ASR MODEL
# ============================================================

from transformers import pipeline
import time

print("=" * 60)
print("🎤 Loading Automatic Speech Recognition Model...")
print("=" * 60)

start_time = time.time()

# ------------------------------------------------------------
# LOAD SUNBIRD WHISPER MODEL
# ------------------------------------------------------------

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL_ID,
    dtype=TORCH_DTYPE,
    device=DEVICE
)

load_time = time.time() - start_time

print(f"✅ ASR Model Loaded Successfully!")
print(f"📦 Model : {ASR_MODEL_ID}")
print(f"🖥️ Device : {DEVICE}")
print(f"⏱️ Load Time : {load_time:.2f} seconds")
print("=" * 60)

🎤 Loading Automatic Speech Recognition Model...


config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

✅ ASR Model Loaded Successfully!
📦 Model : Sunbird/asr-whisper-large-v3-salt
🖥️ Device : cuda
⏱️ Load Time : 128.13 seconds


In [ ]:
# ============================================================
# CELL 6: INITIALIZE GOOGLE TRANSLATE
# ============================================================

from deep_translator import GoogleTranslator

print("=" * 60)
print(" Initializing Translation Service...")
print("=" * 60)

try:
    # Create a translator instance
    translator = GoogleTranslator(source='auto', target='en')

    print("Google Translate initialized successfully!")
    print(f" Translation Service : {TRANSLATION_SERVICE}")

except Exception as e:
    print(f" Failed to initialize translator: {e}")

 Initializing Translation Service...
Google Translate initialized successfully!
 Translation Service : Google Translate


In [ ]:
# ============================================================
# CELL 7: LOAD MEDICAL LLM
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import time

print("=" * 60)
print("🩺 Loading Medical Language Model...")
print("=" * 60)

start_time = time.time()

# ------------------------------------------------------------
# MODEL INFORMATION
# ------------------------------------------------------------

LLM_MODEL_ID = "alpha-ai/LLAMA3-3B-Medical-COT"

# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_ID
)

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    dtype=TORCH_DTYPE,
    device_map=DEVICE
)

load_time = time.time() - start_time

print("Medical LLM Loaded Successfully!")
print(f" Model      : {LLM_MODEL_ID}")
print(f" Device     : {DEVICE}")
print(f"⏱ Load Time  : {load_time:.2f} seconds")
print("=" * 60)

🩺 Loading Medical Language Model...


config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Medical LLM Loaded Successfully!
 Model      : alpha-ai/LLAMA3-3B-Medical-COT
 Device     : cuda
⏱ Load Time  : 73.75 seconds


In [ ]:
# ============================================================
# CELL 8: LOAD TEXT-TO-SPEECH MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModelForTextToWaveform
import time

print("=" * 60)
print(" Loading Text-to-Speech Model...")
print("=" * 60)

start_time = time.time()

# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

tts_tokenizer = AutoTokenizer.from_pretrained(
    TTS_MODEL_ID
)

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

tts_model = AutoModelForTextToWaveform.from_pretrained(
    TTS_MODEL_ID
)

# ------------------------------------------------------------
# MOVE MODEL TO DEVICE
# ------------------------------------------------------------

tts_model.to(DEVICE)

load_time = time.time() - start_time

print("Text-to-Speech Model Loaded Successfully!")
print(f" Model      : {TTS_MODEL_ID}")
print(f" Device     : {DEVICE}")
print(f"⏱ Load Time  : {load_time:.2f} seconds")
print("=" * 60)

 Loading Text-to-Speech Model...


config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Text-to-Speech Model Loaded Successfully!
 Model      : facebook/mms-tts-swh
 Device     : cuda
⏱ Load Time  : 5.19 seconds


In [ ]:
# ============================================================
# CELL 9: SPEECH-TO-TEXT FUNCTION
# ============================================================

def transcribe_audio(audio_path):
    """
    Transcribes an audio file using the loaded ASR model.

    Parameters
    ----------
    audio_path : str
        Path to the uploaded audio file.

    Returns
    -------
    str
        Transcribed text.
    """

    try:
        print("🎤 Transcribing audio...")

        result = asr_pipe(
            audio_path,
            chunk_length_s=30,
            batch_size=8,
            return_timestamps=False
        )

        transcript = result["text"].strip()

        print(" Transcription complete.")

        return transcript

    except Exception as e:
        print(f" Transcription Error: {e}")
        return None

In [ ]:
# ============================================================
# CELL 10: TRANSLATE TEXT
# ============================================================

def translate_text(text, target_language="en"):
    """
    Translates text using Google Translate.

    Parameters
    ----------
    text : str
        Text to translate.

    target_language : str
        Target language code (default = English).

    Returns
    -------
    str
        Translated text.
    """

    try:

        # Skip empty text
        if not text or text.strip() == "":
            return ""

        print("🌍 Translating text...")

        translator = GoogleTranslator(
            source="auto",
            target=target_language
        )

        translated_text = translator.translate(text)

        print("✅ Translation complete.")

        return translated_text

    except Exception as e:
        print(f"❌ Translation Error: {e}")
        return text

In [ ]:
# ============================================================
# CELL 11A: MEDICAL REPORT GENERATOR
# ============================================================

import torch


def generate_medical_report(translated_text):
    """
    Generates a natural-language medical report.

    Parameters
    ----------
    translated_text : str
        English translation from the translation model.

    Returns
    -------
    str
        Natural-language medical report.
    """

    if translated_text is None or translated_text.strip() == "":

        return ""

    print("🩺 Generating medical report...")

    prompt = f"""
You are an experienced medical AI assistant.

A patient reports the following symptoms:

{translated_text}

Write a concise medical report for a healthcare worker.

Your report should include:

- A brief summary of the patient's symptoms.
- The single most likely medical condition.
- Three practical recommendations.
- Whether the condition appears Low, Moderate or High urgency.
- Finish with a short medical disclaimer.

Do not use JSON.
Do not use XML.
Do not use bullet numbering.
Write naturally in complete sentences.
"""

    try:

        inputs = llm_tokenizer(

            prompt,

            return_tensors="pt"

        )

        inputs = {

            k: v.to(llm_model.device)

            for k, v in inputs.items()

        }

        with torch.no_grad():

            outputs = llm_model.generate(

                **inputs,

                max_new_tokens=220,

                do_sample=False,

                temperature=0.2,

                repetition_penalty=1.15,

                no_repeat_ngram_size=3,

                pad_token_id=llm_tokenizer.eos_token_id,

                eos_token_id=llm_tokenizer.eos_token_id

            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

        report = llm_tokenizer.decode(

            generated_tokens,

            skip_special_tokens=True

        ).strip()

        print("\n" + "=" * 80)
        print("MEDICAL REPORT")
        print("=" * 80)
        print(report)
        print("=" * 80)

        return report

    except Exception as e:

        print("\n❌ Medical Report Generation Failed")
        print(e)

        return ""

In [ ]:
# ============================================================
# CELL 11B: INTELLIGENT MEDICAL REPORT PARSER
# ============================================================

import re


def parse_medical_report(report):
    """
    Converts a natural-language medical report into a structured dictionary.

    Parameters
    ----------
    report : str

    Returns
    -------
    dict
    """

    # --------------------------------------------------------
    # Default Output
    # --------------------------------------------------------

    default_output = {

        "summary": "Medical summary unavailable.",

        "possible_condition": "No likely condition identified.",

        "recommendations": [
            "Consult a qualified healthcare professional."
        ],

        "urgency": "Unknown",

        "disclaimer":
        "This AI-generated response is for informational purposes only and "
        "is not a substitute for professional medical advice."

    }

    if report is None or report.strip() == "":

        return default_output

    report = report.strip()

    # --------------------------------------------------------
    # Clean text
    # --------------------------------------------------------

    report = re.sub(r"<.*?>", "", report)

    report = re.sub(r"\s+", " ", report)

    # --------------------------------------------------------
    # Split into sentences
    # --------------------------------------------------------

    sentences = re.split(r'(?<=[.!?])\s+', report)

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    summary = ""

    if len(sentences) >= 2:

        summary = " ".join(sentences[:2])

    elif len(sentences) == 1:

        summary = sentences[0]

    else:

        summary = default_output["summary"]

    # --------------------------------------------------------
    # POSSIBLE CONDITION
    # --------------------------------------------------------

    diagnosis_keywords = [

        "most consistent with",

        "consistent with",

        "suggestive of",

        "likely",

        "probable",

        "diagnosis",

        "compatible with",

        "indicative of",

        "appears to be",

        "is most likely"

    ]

    possible_condition = default_output["possible_condition"]

    for sentence in sentences:

        lower = sentence.lower()

        for keyword in diagnosis_keywords:

            if keyword in lower:

                idx = lower.find(keyword)

                condition = sentence[idx + len(keyword):]

                condition = re.split(r"[.,;]", condition)[0]

                condition = condition.strip(" :-")

                if len(condition) > 3:

                    possible_condition = condition

                    break

        if possible_condition != default_output["possible_condition"]:

            break

    # --------------------------------------------------------
    # RECOMMENDATIONS
    # --------------------------------------------------------

    recommendation_keywords = [

        "should",

        "recommend",

        "recommended",

        "advise",

        "advised",

        "consult",

        "visit",

        "avoid",

        "maintain",

        "take",

        "drink",

        "rest",

        "monitor",

        "seek",

        "follow up"

    ]

    recommendations = []

    for sentence in sentences:

        lower = sentence.lower()

        for keyword in recommendation_keywords:

            if keyword in lower:

                cleaned = sentence.strip()

                cleaned = cleaned.rstrip(". ")

                recommendations.append(cleaned)

                break

    # Remove duplicates

    recommendations = list(dict.fromkeys(recommendations))

    if len(recommendations) == 0:

        recommendations = default_output["recommendations"]

    # --------------------------------------------------------
    # URGENCY
    # --------------------------------------------------------

    urgency = "Unknown"

    lower_report = report.lower()

    if any(x in lower_report for x in [

        "medical emergency",

        "emergency",

        "immediate attention",

        "go to the emergency department",

        "high urgency"

    ]):

        urgency = "High"

    elif any(x in lower_report for x in [

        "moderately urgent",

        "moderate urgency",

        "prompt evaluation",

        "should be evaluated soon"

    ]):

        urgency = "Moderate"

    elif any(x in lower_report for x in [

        "low urgency",

        "routine",

        "non urgent",

        "can be monitored"

    ]):

        urgency = "Low"

    # --------------------------------------------------------
    # DISCLAIMER
    # --------------------------------------------------------

    disclaimer = default_output["disclaimer"]

    for sentence in reversed(sentences):

        if "informational" in sentence.lower():

            disclaimer = sentence.strip()

            break

    # --------------------------------------------------------
    # Debug
    # --------------------------------------------------------

    print("\n" + "=" * 80)

    print("PARSED MEDICAL REPORT")

    print("=" * 80)

    print("SUMMARY")
    print(summary)

    print("\nPOSSIBLE CONDITION")
    print(possible_condition)

    print("\nRECOMMENDATIONS")
    for r in recommendations:
        print("•", r)

    print("\nURGENCY")
    print(urgency)

    print("=" * 80)

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return {

        "summary": summary,

        "possible_condition": possible_condition,

        "recommendations": recommendations,

        "urgency": urgency,

        "disclaimer": disclaimer

    }

In [ ]:
# ============================================================
# CELL 12: TEXT-TO-SPEECH FUNCTION
# ============================================================

import scipy.io.wavfile as wavfile
import torch
import tempfile

def text_to_speech(text):
    """
    Converts text to speech.

    Parameters
    ----------
    text : str
        Text to synthesize.

    Returns
    -------
    str
        Path to generated audio file.
    """

    try:

        if not text.strip():
            return None

        print("🔊 Generating speech...")

        inputs = tts_tokenizer(
            text,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(tts_model.device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            waveform = tts_model(**inputs).waveform

        waveform = waveform.squeeze().cpu().numpy()

        temp_audio = tempfile.NamedTemporaryFile(
            suffix=".wav",
            delete=False
        )

        wavfile.write(
            temp_audio.name,
            rate=16000,
            data=waveform
        )

        print("✅ Speech generated.")

        return temp_audio.name

    except Exception as e:

        print(f"❌ TTS Error: {e}")

        return None

In [ ]:
# ============================================================
# CELL 13 : MASTER PIPELINE
# ============================================================

import time
import traceback
import gradio as gr


def process_audio(audio_path, progress=gr.Progress(track_tqdm=True)):
    """
    Complete Medical Speech Analysis Pipeline

    Steps
    -----
    1. Speech Recognition
    2. Translation
    3. Medical Report Generation
    4. Medical Report Parsing
    5. Speech Synthesis
    """

    start_time = time.time()

    progress(0.0, desc="Initializing...")

    results = {

        "success": False,

        "transcript": "",

        "translation": "",

        "medical_report": "",

        "summary": "",

        "possible_condition": "",

        "recommendations": "",

        "urgency": "",

        "disclaimer": "",

        "tts_audio": None,

        "processing_time": "",

        "status": "",

        "error": ""

    }

    try:

        # =====================================================
        # Validate Input
        # =====================================================

        if audio_path is None:

            results["error"] = "No audio uploaded."

            return results

        print("\n" + "=" * 70)
        print("🩺 MULTILINGUAL MEDICAL SPEECH ASSISTANT")
        print("=" * 70)

        # =====================================================
        # STEP 1 : Speech Recognition
        # =====================================================

        progress(
            0.10,
            desc="🎤 Transcribing audio..."
        )

        transcript = transcribe_audio(audio_path)

        if transcript is None or transcript.strip() == "":

            raise Exception("Speech recognition failed.")

        results["transcript"] = transcript

        print("✓ Speech Recognition Complete")

        # =====================================================
        # STEP 2 : Translation
        # =====================================================

        progress(
            0.30,
            desc="🌍 Translating..."
        )

        translated = translate_text(
            transcript,
            target_language="en"
        )

        results["translation"] = translated

        print("✓ Translation Complete")

        # =====================================================
        # STEP 3 : Medical Report Generation
        # =====================================================

        progress(
            0.55,
            desc="🩺 Generating medical report..."
        )

        report = generate_medical_report(
            translated
        )

        results["medical_report"] = report

        print("✓ Medical Report Generated")

        # =====================================================
        # STEP 4 : Parse Report
        # =====================================================

        progress(
            0.70,
            desc="📄 Parsing medical report..."
        )

        medical = parse_medical_report(
            report
        )

        results["summary"] = medical["summary"]

        results["possible_condition"] = medical["possible_condition"]

        results["recommendations"] = "\n• " + "\n• ".join(
            medical["recommendations"]
        )

        results["urgency"] = medical["urgency"]

        results["disclaimer"] = medical["disclaimer"]

        print("✓ Medical Report Parsed")

        # =====================================================
        # STEP 5 : Text To Speech
        # =====================================================

        progress(
            0.90,
            desc="🔊 Generating speech..."
        )

        speech = f"""
Medical Summary

{results['summary']}

Possible Condition

{results['possible_condition']}

Recommendations

{results['recommendations']}

Urgency

{results['urgency']}
"""

        results["tts_audio"] = text_to_speech(
            speech
        )

        print("✓ Speech Synthesis Complete")

        # =====================================================
        # FINISH
        # =====================================================

        elapsed = round(
            time.time() - start_time,
            2
        )

        progress(
            1.0,
            desc="✅ Completed"
        )

        results["processing_time"] = elapsed

        results["status"] = "Analysis completed successfully."

        results["success"] = True

        print("=" * 70)
        print("🎉 PIPELINE FINISHED")
        print(f"Processing Time : {elapsed} seconds")
        print("=" * 70)

        return results

    except Exception as e:

        traceback.print_exc()

        elapsed = round(
            time.time() - start_time,
            2
        )

        results["processing_time"] = elapsed

        results["status"] = "Pipeline failed."

        results["error"] = str(e)

        progress(
            1.0,
            desc="❌ Failed"
        )

        return results

In [ ]:
import gradio as gr
print(gr.__version__)

6.20.0


In [ ]:
# ------------------------------------------------------------
# Interface
# ------------------------------------------------------------

with gr.Blocks(
    title="Multilingual Medical Speech Assistant",
    theme=gr.themes.Soft()
) as demo:

    # ------------------------------------------------------------
    # Custom CSS
    # ------------------------------------------------------------

    gr.HTML("""
    <style>

    /* ==========================================================
       PAGE
    ========================================================== */

    .gradio-container{
        max-width:1400px !important;
        width:96% !important;
        margin:auto !important;
    }

    /* ==========================================================
       TITLE
    ========================================================== */

    h1{
        font-size:32px !important;
        font-weight:700 !important;
        margin-bottom:8px !important;
    }

    /* ==========================================================
       INTRODUCTION
    ========================================================== */

    .prose{
        font-size:14px !important;
        line-height:1.35 !important;
    }

    .prose p{
        margin-bottom:6px !important;
    }

    .prose li{
        font-size:14px !important;
        margin-bottom:2px !important;
    }

    /* ==========================================================
       TAB TITLES
    ========================================================== */

    button[role="tab"]{
        font-size:14px !important;
        font-weight:600 !important;
    }

    /* ==========================================================
       LABELS
    ========================================================== */

    label{
        font-size:14px !important;
        font-weight:600 !important;
    }

    /* ==========================================================
       TEXTBOXES
    ========================================================== */

    textarea,
    input{
        font-size:14px !important;
    }

    /* ==========================================================
       BUTTONS
    ========================================================== */

    button{
        font-size:14px !important;
        min-height:42px !important;
    }

    /* ==========================================================
       FOOTER
    ========================================================== */

    footer{
        display:none !important;
    }

    </style>
    """)

    gr.Markdown("""
# 🩺 Multilingual Medical Speech Assistant

Upload or record your voice in a supported language.

The system will:

1. 🎤 Transcribe your speech
2. 🌍 Translate it to English
3. 🩺 Generate a medical recommendation
4. 🔊 Read the recommendation aloud

⚠️ This system is for educational purposes and should not replace professional medical advice.
""")

    # --------------------------------------------------------
    # INPUT
    # --------------------------------------------------------

    with gr.Row():

        audio_input = gr.Audio(
            sources=["upload", "microphone"],
            type="filepath",
            label="🎤 Upload or Record Audio",
            scale=5
        )

        analyze_button = gr.Button(
            "🚀 Analyze",
            variant="primary",
            scale=1,
            min_width=140
        )

    # --------------------------------------------------------
    # OUTPUTS
    # --------------------------------------------------------

    with gr.Tab("📝 Transcript"):

        transcript = gr.Textbox(
            lines=4,
            label="Speech Transcript"
        )

    with gr.Tab("🌍 Translation"):

        translation = gr.Textbox(
            lines=4,
            label="English Translation"
        )

    with gr.Tab("🩺 Medical Report"):

        summary = gr.Textbox(
            lines=3,
            label="Medical Summary"
        )

        condition = gr.Textbox(
            lines=1,
            label="Most Likely Condition"
        )

        recommendations = gr.Textbox(
            lines=4,
            label="Recommendations"
        )

        urgency = gr.Textbox(
            lines=1,
            label="Urgency Level"
        )

        disclaimer = gr.Textbox(
            lines=2,
            label="Medical Disclaimer"
        )

    with gr.Tab("🔊 Spoken Response"):

        audio_output = gr.Audio(
            label="Generated Speech",
            autoplay=False
        )

    status = gr.Textbox(
        label="Processing Status",
        lines=1
    )

    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    analyze_button.click(
        fn=run_pipeline,
        inputs=audio_input,
        outputs=[
            transcript,
            translation,
            summary,
            condition,
            recommendations,
            urgency,
            audio_output,
            disclaimer,
            status
        ]
    )

print("✅ Web application created successfully!")

✅ Web application created successfully!


In [ ]:
# ============================================================
# CELL 15: LAUNCH WEB APPLICATION
# ============================================================

demo.launch(
    debug=True,
    share=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://009e61afbba8edd9f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:7860")
    print("Status:", r.status_code)
    print("Local Gradio server is running.")
except Exception as e:
    print(e)

Status: 200
Local Gradio server is running.
